Notebook RAG

In [1]:
! pip install langchain langchain-core langchain-community langchain-google-genai chromadb bs4 jq


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#from langchain import hub
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate, format_document
from langchain_community.document_loaders import WebBaseLoader, JSONLoader
from langchain_community.vectorstores import Chroma

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from dotenv import load_dotenv

In [7]:
import os
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [8]:
print(GOOGLE_API_KEY)

In [10]:
loader = JSONLoader(file_path='../data/recettes_details_sample.json',jq_schema='.[]', text_content=False)
docs = loader.load()

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [12]:
import html
import time
from langchain_community.vectorstores import Chroma

for doc in docs:
    doc.page_content = html.unescape(doc.page_content)

ids = [f"{doc.metadata.get('source', 'recette')}_{doc.metadata.get('seq_num', idx)}" for idx, doc in enumerate(docs)]

vectorstore = Chroma(embedding_function=gemini_embeddings, persist_directory="./chroma_db")

batch_size = 10

# Récupère les IDs déjà indexés pour reprendre où on s'est arrêté
deja_indexes = set(vectorstore._collection.get()["ids"])
print(f"Déjà indexés : {len(deja_indexes)} documents")

print(f"Indexation de {len(docs)} documents...")

for i in range(0, len(docs), batch_size):
    batch_docs = docs[i : i + batch_size]
    batch_ids  = ids[i : i + batch_size]

    # Skip si tous les docs du lot sont déjà dans Chroma
    if all(bid in deja_indexes for bid in batch_ids):
        print(f"Lot {i//batch_size + 1} déjà indexé, on passe")
        continue

    try:
        vectorstore.add_documents(batch_docs, ids=batch_ids)
        print(f"Lot {i//batch_size + 1} ajouté ({min(i + batch_size, len(docs))}/{len(docs)})")
        if i + batch_size < len(docs):
            time.sleep(12)
    except Exception as e:
        print(f"⚠️ Quota atteint au lot {i//batch_size + 1} — relance la cellule dans 1 min, elle reprendra ici")
        break

print("Terminé !")

C:\Users\ellen\AppData\Local\Temp\ipykernel_28844\1811853155.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(embedding_function=gemini_embeddings, persist_directory="./chroma_db")


Déjà indexés : 0 documents
Indexation de 203 documents...
Lot 1 ajouté (10/203)
Lot 2 ajouté (20/203)
Lot 3 ajouté (30/203)
Lot 4 ajouté (40/203)
Lot 5 ajouté (50/203)
Lot 6 ajouté (60/203)
Lot 7 ajouté (70/203)
Lot 8 ajouté (80/203)
Lot 9 ajouté (90/203)
Lot 10 ajouté (100/203)
Lot 11 ajouté (110/203)
Lot 12 ajouté (120/203)
Lot 13 ajouté (130/203)
⚠️ Quota atteint au lot 14 — relance la cellule dans 1 min, elle reprendra ici
Terminé !


In [13]:
import shutil

# Zippe le dossier chroma_db
shutil.make_archive("chroma_db_backup", "zip", ".", "chroma_db")
print("✅ chroma_db_backup.zip créé")


✅ chroma_db_backup.zip créé


In [14]:
# Load from disk
vectorstore_disk = Chroma(
                        persist_directory="./chroma_db",       # Directory of db
                        embedding_function=gemini_embeddings   # Embedding model
                   )
# Get the Retriever interface for the store to use later.
# When an unstructured query is given to a retriever it will return documents.
# Read more about retrievers in the following link.
# https://python.langchain.com/docs/modules/data_connection/retrievers/
#
# Since only 1 document is stored in the Chroma vector store, search_kwargs `k`
# is set to 1 to decrease the `k` value of chroma's similarity search from 4 to
# 1. If you don't pass this value, you will get a warning.
retriever = vectorstore_disk.as_retriever(search_kwargs={"k": 6})

# Check if the retriever is working by trying to fetch the relevant docs related
resultats = retriever.invoke("pizza")
print(f"Nombre de documents trouvés : {len(resultats)}")

Nombre de documents trouvés : 6


In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

# To configure model parameters use the `generation_config` parameter.
# eg. generation_config = {"temperature": 0.7, "topP": 0.8, "topK": 40}
# If you only want to set a custom temperature for the model use the
# "temperature" parameter directly.

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [16]:
# Prompt template to query Gemini
llm_prompt_template = """You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you don't know the answer, just say that you don't know.
Use five sentences maximum and keep the answer concise.\n
Question: {question} \nContext: {context} \nAnswer:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

# """Tu es un chef cuisinier expert et un assistant de recommandation culinaire qui s'appelle Duflan.
# Ton objectif est de proposer la meilleure recette possible en te basant STRICTEMENT sur le contexte fourni.

# Analyse attentivement la "Question" de l'utilisateur pour y identifier ses trois contraintes :
# 1. Les ingrédients dont il dispose.
# 2. Le temps de préparation dont il dispose.
# 3. Ses préférences gustatives ou ses plats favoris.

# Sélectionne dans le contexte la recette qui respecte au mieux ces critères, en priorisant l'utilisation des ingrédients disponibles.
# Ne modifie jamais les ingrédients ou la préparation d'une recette
# Si aucune recette du contexte ne correspond parfaitement, choisis celle qui s'en rapproche le plus et précise clairement quels ingrédients manquent ou quels ajustements ont été faits.
# Si le contexte ne contient aucune recette pertinente, réponds poliment que tu n'as pas de recette adaptée pour le moment. N'invente jamais de recette hors du contexte.

# Format de sortie attendue (important):
# **Nom de la recette** (Temps estimé)
# * **Ingrédients utilisés :** [Liste]
# * **Ingrédients manquants (à acheter) :** [Liste ou "Aucun"]
# * **Pourquoi cette recette ?** [Une courte phrase expliquant comment elle répond à la question de l'utilisateur]
# * **Instructions :**
# 1. [Étape 1]
# 2. [Étape 2]
# ...

# Question : {question}
# Contexte : {context}

# Réponse :"""

input_variables=['context', 'question'] input_types={} partial_variables={} template="You are an assistant for question-answering tasks.\nUse the following context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse five sentences maximum and keep the answer concise.\n\nQuestion: {question} \nContext: {context} \nAnswer:"


In [17]:
# Combine data from documents to readable string format.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create stuff documents chain using LCEL.
#
# This is called a chain because you are chaining together different elements
# with the LLM. In the following example, to create the stuff chain, you will
# combine the relevant context from the website data matching the question, the
# LLM model, and the output parser together like a chain using LCEL.
#
# The chain implements the following pipeline:
# 1. Extract the website data relevant to the question from the Chroma
#    vector store and save it to the variable `context`.
# 2. `RunnablePassthrough` option to provide `question` when invoking
#    the chain.
# 3. The `context` and `question` are then passed to the prompt where they
#    are populated in the respective variables.
# 4. This prompt is then passed to the LLM (`gemini-2.0-flash`).
# 5. Output from the LLM is passed through an output parser
#    to structure the model's response.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

In [18]:
from IPython.display import Markdown

Markdown(rag_chain.invoke("J'ai des tomates, j'ai 20 minutes, et j'adore la cuisine italienne"))

Vous pouvez préparer la recette italienne de la **"Sauce tomate il sugo"**. Cette recette très facile ne demande que 10 minutes de préparation avec une cuisson rapide à la poêle, ce qui s'adapte parfaitement à vos 20 minutes. Elle met en valeur vos tomates avec des ingrédients simples comme de l'ail, de l'oignon, du basilic frais et de l'huile d'olive. C'est l'option idéale pour savourer un classique de la cuisine italienne en un minimum de temps.

In [19]:
Markdown(rag_chain.invoke("Donne moi une recette pour faire une galette St Michel"))

Je ne sais pas. Le contexte fourni ne contient pas de recette pour faire une galette St Michel.